# 02 - Sanity Checks

Quick checks that the `gpt2_chatbot` package works end to end:

1. Model builds and a forward pass returns logits of the expected shape.
2. Parameter count is in the right ballpark for GPT-2 small (~124M / ~163M with untied head).
3. The causal mask prevents attending to future tokens.
4. Tokenizer round-trips text.
5. Greedy and sampled generation both run.
6. (Optional) Load official OpenAI GPT-2 weights and generate coherent text.

This notebook only reads from the package; it does not modify any source files.

In [ ]:
import sys
from pathlib import Path

# Make the src-layout package importable when running from notebooks/.
SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import torch

from gpt2_chatbot.model import GPTModel, get_config
from gpt2_chatbot.tokenizer import get_tokenizer, text_to_token_ids, token_ids_to_text
from gpt2_chatbot.inference import generate, generate_text_simple

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## 1. Build model + forward-pass shape check

In [ ]:
cfg = get_config("gpt2-small (124M)", context_length=256)
model = GPTModel(cfg).to(device).eval()

batch, seq_len = 2, 8
dummy = torch.randint(0, cfg["vocab_size"], (batch, seq_len), device=device)
logits = model(dummy)

expected = (batch, seq_len, cfg["vocab_size"])
assert logits.shape == expected, f"got {tuple(logits.shape)}, expected {expected}"
print("OK - logits shape:", tuple(logits.shape))

## 2. Parameter count

GPT-2 small has ~124M parameters with the token embedding tied to the output head. Here the head is a separate `nn.Linear`, so the raw total is higher (~163M). We report both.

In [ ]:
total = sum(p.numel() for p in model.parameters())
tied = total - model.out_head.weight.numel()  # subtract the untied output head
print(f"Total params:            {total:,}")
print(f"With tied embeddings:    {tied:,}")
assert 120_000_000 < tied < 130_000_000, "tied param count outside expected GPT-2 small range"
print("OK - parameter count in expected range")

## 3. Causal mask check

Changing a token should only affect logits at that position and later ones, never earlier positions.

In [ ]:
seq_len = 6
ids = torch.randint(0, cfg["vocab_size"], (1, seq_len), device=device)
with torch.no_grad():
    base = model(ids)

# Flip the token at the last position.
ids_mod = ids.clone()
ids_mod[0, -1] = (ids_mod[0, -1] + 1) % cfg["vocab_size"]
with torch.no_grad():
    changed = model(ids_mod)

earlier_diff = (base[:, :-1, :] - changed[:, :-1, :]).abs().max().item()
last_diff = (base[:, -1, :] - changed[:, -1, :]).abs().max().item()
print(f"max change at earlier positions: {earlier_diff:.3e} (should be ~0)")
print(f"max change at last position:     {last_diff:.3e} (should be > 0)")
assert earlier_diff < 1e-5, "future token leaked into earlier positions - causal mask broken"
assert last_diff > 0, "changing the last token had no effect"
print("OK - attention is causal")

## 4. Tokenizer round-trip

In [ ]:
tok = get_tokenizer()
text = "Every effort moves you"
ids = text_to_token_ids(text, tok)
roundtrip = token_ids_to_text(ids, tok)
print("ids:", ids.tolist())
print("roundtrip:", repr(roundtrip))
assert roundtrip == text, "tokenizer did not round-trip"
print("OK - tokenizer round-trips")

## 5. Generation runs (random weights - output will be gibberish)

In [ ]:
start = text_to_token_ids("Every effort moves you", tok).to(device)

greedy = generate_text_simple(model, start, max_new_tokens=10, context_size=cfg["context_length"])
sampled = generate(model, start, max_new_tokens=10, context_size=cfg["context_length"],
                   top_k=25, temperature=1.4)

print("greedy :", repr(token_ids_to_text(greedy, tok)))
print("sampled:", repr(token_ids_to_text(sampled, tok)))
assert greedy.shape[1] == start.shape[1] + 10
assert sampled.shape[1] <= start.shape[1] + 10
print("OK - both generation paths run")

## 6. (Optional) Load pretrained GPT-2 weights

This requires the `gpt_download` helper and `tensorflow` to fetch the official OpenAI checkpoints. If they aren't available, the cell skips gracefully. With real weights, generation should produce coherent English.

In [ ]:
try:
    from gpt_download import download_and_load_gpt2  # provided by LLMs-from-scratch
    from gpt2_chatbot.model import load_weights_into_gpt

    pretrained_cfg = get_config("gpt2-small (124M)", context_length=1024, qkv_bias=True)
    settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

    gpt = GPTModel(pretrained_cfg)
    load_weights_into_gpt(gpt, params)
    gpt.to(device).eval()

    out = generate(
        gpt,
        text_to_token_ids("Every effort moves you", tok).to(device),
        max_new_tokens=25,
        context_size=pretrained_cfg["context_length"],
        top_k=50,
        temperature=1.0,
    )
    print(token_ids_to_text(out, tok))
except Exception as e:
    print("Skipping pretrained-weight check:", type(e).__name__, e)